# Evaluate AutoScientist adapters on a Colab T4
Runtime > Change runtime type > **T4 GPU** before running. Put your adapters in Google Drive under `adapters/<run>/`.

In [ ]:
!nvidia-smi | head -8
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
# 1. get the code (replace with your GitHub URL)
!git clone https://github.com/<GITHUB_USERNAME>/autoscientist-chart-qa.git
%cd autoscientist-chart-qa
!pip install -q -U bitsandbytes peft transformers accelerate

In [ ]:
# 2. copy adapters from Drive (adjust the path), then list their real settings
from google.colab import drive; drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/adapters/* adapters/
!python scripts/adapter_info.py

In [ ]:
# 3. predict for every adapter folder that has no predictions yet, plus the untuned base
import os, subprocess
MODEL = 'Qwen/Qwen3-4B-Base'
os.makedirs('runs/preds', exist_ok=True)
runs = [(None, 'base_qwen3-4b')] + [(d, d) for d in sorted(os.listdir('adapters')) if os.path.isdir(f'adapters/{d}')]
for adapter, name in runs:
    out = f'runs/preds/{name}_preds.jsonl'
    if os.path.exists(out): print('skip', name); continue
    cmd = ['python','scripts/predict_local.py','--inp','data/eval.jsonl','--out',out,'--model',MODEL]
    if adapter: cmd += ['--adapter', f'adapters/{adapter}']
    print('running', name)
    if subprocess.run(cmd).returncode: print('FAILED', name)

In [ ]:
# 4. score everything and show the table
!python scripts/summarize.py

In [ ]:
# 5. save predictions back to Drive
!cp -r runs /content/drive/MyDrive/runs_backup